# Chapter 5: predict a number, then a class

Both models in the chapter: RDF from wort sugars (regression), stuck fermentation risk (classification) and the outlet reorder list. Synthetic data only; no real brewery's numbers.

What to look at: every model against its baseline.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import confusion_matrix, mean_absolute_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor

## 5.2 RDF from wort sugars

180 batches: 144 to learn from, 36 held back. The baseline always predicts the training average.

In [ ]:
rdf = pd.read_csv('../data/rdf_batches.csv')
X = ['glucose_pct', 'maltose_pct', 'maltotriose_pct', 'mash_temp_c', 'og_plato']
train, test = rdf[rdf.split == 'train'], rdf[rdf.split == 'test']
baseline = train.rdf_pct.mean()
model = LinearRegression().fit(train[X], train.rdf_pct)
test = test.assign(model=model.predict(test[X]).round(1))
print('Baseline (training average):', round(baseline, 1))
print(test.head(5)[['batch_id', 'rdf_pct', 'model']])
print('Model MAE:', round(mean_absolute_error(test.rdf_pct, model.predict(test[X])), 1))
print('Baseline MAE:', round((test.rdf_pct - baseline).abs().mean(), 1))

In [ ]:
plt.scatter(test.rdf_pct, test.model)
plt.plot([60, 74], [60, 74], '--')
plt.axhline(baseline, color='orange')
plt.xlabel('Measured RDF (%)'); plt.ylabel('Predicted RDF (%)')
plt.show()

## 5.4 Overfitting

A tree with no depth limit memorises its training rows. Compare its training and test error.

In [ ]:
tree = DecisionTreeRegressor(random_state=0).fit(train[X], train.rdf_pct)
print('Tree, training MAE:', round(mean_absolute_error(train.rdf_pct, tree.predict(train[X])), 2))
print('Tree, test MAE:', round(mean_absolute_error(test.rdf_pct, tree.predict(test[X])), 2))

## 5.3 Stuck fermentation risk

Train on the oldest 360 batches, test on the newest 120. The model gives a risk score; a threshold turns it into a flag.

In [ ]:
s = pd.read_csv('../data/stuck_ferments.csv')
F = ['pitch_rate_m_cells_ml', 'viability_pct', 'wort_do_mg_l', 'og_plato', 'ferm_temp_c', 'gravity_drop_24h_plato']
tr, te = s[s.split == 'train'], s[s.split == 'test']
clf = make_pipeline(StandardScaler(), LogisticRegression()).fit(tr[F], tr.stuck == 'stuck')
risk = clf.predict_proba(te[F])[:, 1]
actual = te.stuck == 'stuck'
for th in (0.5, 0.3):
    flag = risk > th
    tn, fp, fn, tp = confusion_matrix(actual, flag).ravel()
    print(f'threshold {th}: hits {tp}, false alarms {fp}, misses {fn}, clear {tn}')
    print(f'  accuracy {(tp + tn) / len(te):.0%}, precision {tp / (tp + fp):.0%}, recall {tp / (tp + fn):.0%}')
print('Never flag anything, accuracy:', f'{(~actual).mean():.0%}')

Write the cost of each mistake down before you choose the threshold.

In [ ]:
false_alarm_h, miss_h = 0.5, 3 * 24
print('5 false alarms:', 5 * false_alarm_h, 'brewer hours')
print('2 misses:', 2 * miss_h, 'fermenter hours')

## 5.5 Which outlets reorder next month

Out of fold scores: each outlet is scored by a model that never saw it. Then rank by risk of lapsing.

In [ ]:
from sklearn.compose import make_column_transformer
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import OneHotEncoder

o = pd.read_csv('../data/outlets.csv')
num = ['months_since_first_order', 'orders_last_3m', 'days_since_last_order', 'sku_count', 'rep_visits_60d']
pre = make_column_transformer((OneHotEncoder(), ['outlet_type']), (StandardScaler(), num))
m = make_pipeline(pre, LogisticRegression(max_iter=1000))
lapsed = (o.reordered == 'no').astype(int)
risk = cross_val_predict(m, o, lapsed, cv=StratifiedKFold(5, shuffle=True, random_state=0), method='predict_proba')[:, 1]
top = o.iloc[np.argsort(-risk, kind='stable')[:100]]
print('Outlets:', len(o), 'reordered:', (o.reordered == 'yes').sum())
print('Lapsed among the 100 highest risk:', (top.reordered == 'no').sum())
print('Expected by chance:', round(100 * lapsed.mean()))